# NBA Game Prediction: Data Wrangling

**Goal:** Load the raw NBA game data, check its quality, and produce a clean dataset for exploratory analysis and modeling.

**Data:** Team-level game stats for three NBA regular seasons, one row per team per game. Used NBA stats Api

**This notebook:** loads the raw data, checks for missing values and duplicates, fixes data types (such as dates), and saves the cleaned file as `nba_games_cleaned.csv`, which the EDA notebook uses.

In [1]:
#import packages
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
!pip install nba-api


In [3]:
from nba_api.stats.endpoints import LeagueGameLog
help(LeagueGameLog)

Help on class LeagueGameLog in module nba_api.stats.endpoints.leaguegamelog:

class LeagueGameLog(nba_api.stats.endpoints._base.Endpoint)
 |  LeagueGameLog(
 |      counter=0,
 |      direction='ASC',
 |      league_id='00',
 |      player_or_team_abbreviation='T',
 |      season='2026-27',
 |      season_type_all_star='Regular Season',
 |      sorter='DATE',
 |      date_from_nullable='',
 |      date_to_nullable='',
 |      proxy=None,
 |      headers=None,
 |      timeout=30,
 |      get_request=True
 |  )
 |
 |  Method resolution order:
 |      LeagueGameLog
 |      nba_api.stats.endpoints._base.Endpoint
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(
 |      self,
 |      counter=0,
 |      direction='ASC',
 |      league_id='00',
 |      player_or_team_abbreviation='T',
 |      season='2026-27',
 |      season_type_all_star='Regular Season',
 |      sorter='DATE',
 |      date_from_nullable='',
 |      date_to_nullable='',
 |      proxy=None,
 |      headers

In [4]:
#get the data from the last three seasons 
year_2023_24 = LeagueGameLog(season='2023-24',season_type_all_star='Regular Season').get_data_frames()[0]

year_2024_25 = LeagueGameLog(season='2024-25',season_type_all_star='Regular Season').get_data_frames()[0]

year_2025_26 =  LeagueGameLog(season='2025-26',season_type_all_star='Regular Season').get_data_frames()[0]

In [5]:
#combine the data from the last three regular seansons
nba_games = pd.concat([year_2023_24,year_2024_25,year_2025_26],axis=0)

Since the matchup shows an @ for away games we should try and make a new column specifically for which team is home or away

In [6]:
#make home team column
nba_games['HOME_TEAM'] = nba_games['MATCHUP'].str.contains('vs.' ,case=False,regex = False)
nba_games['HOME_TEAM']

0        True
1        True
2       False
3       False
4       False
        ...  
2455    False
2456    False
2457    False
2458    False
2459    False
Name: HOME_TEAM, Length: 7380, dtype: bool

Opponents also likely has a big impact on if a team wins are uses. So it would be helful to add a new column saying who the opponent is

In [7]:
nba_games['OPPONENT'] = nba_games['MATCHUP'].str.extract(r'(?:vs\.|@)\s*([A-Za-z]+)')
nba_games['OPPONENT']

0       LAL
1       PHX
2       DEN
3       GSW
4       MIA
       ... 
2455    LAL
2456    HOU
2457    CLE
2458    IND
2459    NYK
Name: OPPONENT, Length: 7380, dtype: object

In [8]:
#install fd data profiling
!pip install fg-data-profiling
import data_profiling
from data_profiling import ProfileReport

In [9]:
#get report on the last three seasons
nba_report = ProfileReport(nba_games, title='NBA Game Log Data Report')
nba_report.to_file("nba_games_profile_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████████████████████████████████████| 31/31 [00:00<00:00, 590.28it/s]
/opt/anaconda3/lib/python3.13/site-packages/data_profiling/model/correlations.py:87: UserWarning: There was an attempt to calculate the auto correlation, but this failed.
To hide this warning, disable the calculation
(using `df.profile_report(correlations={"auto": {"calculate": False}})`
If this is problematic for your use case, please report this as an issue:
https://github.com/Data-Centric-AI-Community/data-profiling/issues
(include the error message: 'cannot reindex on an axis with duplicate labels')
  warnings.warn(


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

It looks like the data is already pretty clean, with it showing clear uniform data when it comes to teams. There's still a couple things we can check on like MIN being imbalanced, and also inspecting the video column. There is also missing cells we can look at

In [10]:
#look at missing cell
missing_row = nba_games[nba_games.isnull().any(axis=1)]
missing_row.isnull().any()

SEASON_ID            False
TEAM_ID              False
TEAM_ABBREVIATION    False
TEAM_NAME            False
GAME_ID              False
GAME_DATE            False
MATCHUP              False
WL                   False
MIN                  False
FGM                  False
FGA                  False
FG_PCT               False
FG3M                 False
FG3A                 False
FG3_PCT              False
FTM                  False
FTA                  False
FT_PCT                True
OREB                 False
DREB                 False
REB                  False
AST                  False
STL                  False
BLK                  False
TOV                  False
PF                   False
PTS                  False
PLUS_MINUS           False
VIDEO_AVAILABLE      False
HOME_TEAM            False
OPPONENT             False
dtype: bool

In [11]:
missing_row['FTA']

2348    0
Name: FTA, dtype: int64

In [12]:
#fill the missing value with 0 since there was 0 freethrows taken
nba_games['FT_PCT'] = nba_games['FT_PCT'].fillna(0)

In [13]:
#inspect the MIN
nba_games['MIN'].describe()

count    7380.000000
mean      241.307588
std         6.144925
min       240.000000
25%       240.000000
50%       240.000000
75%       240.000000
max       290.000000
Name: MIN, dtype: float64

In [14]:
nba_games[nba_games['MIN']==290]

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,HOME_TEAM,OPPONENT
373,22023,1610612762,UTA,Utah Jazz,0022300214,2023-11-19,UTA vs. PHX,L,290,48,...,27,11,5,19,27,137,-3,1,True,PHX
383,22023,1610612756,PHX,Phoenix Suns,0022300214,2023-11-19,PHX @ UTA,W,290,50,...,36,11,13,18,28,140,3,1,False,UTA
994,22023,1610612758,SAC,Sacramento Kings,0022300476,2024-01-03,SAC vs. ORL,W,290,51,...,36,9,3,21,21,138,3,1,True,ORL
998,22023,1610612753,ORL,Orlando Magic,0022300476,2024-01-03,ORL @ SAC,L,290,44,...,33,10,7,21,30,135,-3,1,False,SAC
1338,22023,1610612744,GSW,Golden State Warriors,0022300650,2024-01-27,GSW vs. LAL,L,290,55,...,37,11,3,17,34,144,-1,1,True,LAL
1351,22023,1610612747,LAL,Los Angeles Lakers,0022300650,2024-01-27,LAL @ GSW,W,290,49,...,33,9,5,20,21,145,1,1,False,GSW
1473,22023,1610612761,TOR,Toronto Raptors,0022300710,2024-02-04,TOR @ OKC,L,290,48,...,38,6,5,12,16,127,-8,1,False,OKC
1474,22023,1610612760,OKC,Oklahoma City Thunder,0022300710,2024-02-04,OKC vs. TOR,W,290,52,...,40,6,8,11,15,135,8,1,True,TOR
1746,22023,1610612741,CHI,Chicago Bulls,0022300846,2024-02-28,CHI vs. CLE,W,290,44,...,29,6,8,19,18,132,9,1,True,CLE
1748,22023,1610612739,CLE,Cleveland Cavaliers,0022300846,2024-02-28,CLE @ CHI,L,290,45,...,37,7,11,12,25,123,-9,1,False,CHI


Looking up these games it looks like they're from double overtime which makes sense that the minutes category are as long as they are

In [15]:
#get rid of the video avaialable column
nba_games = nba_games.drop(columns='VIDEO_AVAILABLE')

In [15]:
#Final report for verification
nba_final_report = nba_games.profile_report(title="NBA Game Data - Final Cleaned Report")
nba_final_report.to_file("nba_games_profile_final_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|█████████████████████████████████████████| 30/30 [00:00<00:00, 1535.55it/s]
/opt/anaconda3/lib/python3.13/site-packages/data_profiling/model/correlations.py:87: UserWarning: There was an attempt to calculate the auto correlation, but this failed.
To hide this warning, disable the calculation
(using `df.profile_report(correlations={"auto": {"calculate": False}})`
If this is problematic for your use case, please report this as an issue:
https://github.com/Data-Centric-AI-Community/data-profiling/issues
(include the error message: 'cannot reindex on an axis with duplicate labels')
  warnings.warn(


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
nba_games.to_csv('nba_games_cleaned.csv', index=False)